Import libraries

In [10]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [11]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Women/val_women.csv")

CPU times: user 25 ms, sys: 7.09 ms, total: 32.1 ms
Wall time: 31.4 ms
CPU times: user 13.8 ms, sys: 3.99 ms, total: 17.8 ms
Wall time: 17.7 ms
CPU times: user 4.37 ms, sys: 14 μs, total: 4.38 ms
Wall time: 4.36 ms


In [12]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [13]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [14]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [15]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [16]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [17]:
epochs = 150
alpha = 1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 0
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0.0
shared_dropout = 0.0 
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [ ]:
import pandas as pd
import numpy as np
import torch
import io
import json
from datetime import datetime
from pathlib import Path
from contextlib import redirect_stdout
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [5e-5, 1e-4, 5e-4, 1e-3]
wd_grid = [1e-6, 1e-5, 1e-4]
method_grid = ["mmd_linear", "mmd_rbf"]
alpha_grid = [0.1, 0.5, 1.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

# 2. Loop over all hyperparameter combinations
for grid_lr, grid_wd, grid_method, grid_alpha in product(lr_grid, wd_grid, method_grid, alpha_grid):
    run_rows = []

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            alpha=grid_alpha,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        # Suppress internal training logs
        with redirect_stdout(io.StringIO()):
            cfr.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p = cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred = cfr.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "method": grid_method,
            "alpha": grid_alpha,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)

    # Aggregate each hyperparameter combination
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "method": grid_method,
        "alpha": grid_alpha,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

# 4. Save results so they can be reused without retraining
output_dir = project_root / "Conversion" / "CFRnet" / "results"
output_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
ranking_path = output_dir / f"cfrnet_women_grid_{run_id}.csv"
best_path = output_dir / f"cfrnet_women_best_{run_id}.json"
latest_ranking_path = output_dir / "cfrnet_women_grid_latest.csv"
latest_best_path = output_dir / "cfrnet_women_best_latest.json"

df_grid.to_csv(ranking_path, index=False)
df_grid.to_csv(latest_ranking_path, index=False)

best_payload = best_cfg.to_dict()
best_payload = {k: (v.item() if hasattr(v, "item") else v) for k, v in best_payload.items()}
best_payload["saved_at"] = datetime.now().isoformat(timespec="seconds")

with open(best_path, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)
with open(latest_best_path, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)

# Only print final best result
print("Best hyperparameters by mean TEST AUQC:")
print(
    f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
    f"method={best_cfg['method']}, alpha={best_cfg['alpha']:.1f}"
)
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} +- {best_cfg['std_AUQC']:.4f}")
print(f"Saved full grid to: {latest_ranking_path}")
print(f"Saved best config to: {latest_best_path}")

Locked random seed: 412312


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 42


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1874


...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 902745


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Locked random seed: 1
Best hyperparameters by mean TEST AUQC:
  lr=5.0e-04, weight_decay=1.0e-04, method=mmd_rbf, alpha=0.5
  mean TEST AUQC = 0.7682 ± 0.1111


/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


In [4]:
# Quick check: print cached best result without retraining
import json
from pathlib import Path

if 'best_cfg' in globals():
    print("Best hyperparameters by mean TEST AUQC:")
    print(
        f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
        f"method={best_cfg['method']}, alpha={best_cfg['alpha']:.1f}"
    )
    print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} +- {best_cfg['std_AUQC']:.4f}")
else:
    if 'project_root' in globals():
        root = project_root
    else:
        root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")

    latest_best_path = root / "Conversion" / "CFRnet" / "results" / "cfrnet_women_best_latest.json"
    if latest_best_path.exists():
        with open(latest_best_path, "r", encoding="utf-8") as f:
            saved = json.load(f)
        print("Best hyperparameters by mean TEST AUQC (loaded from file):")
        print(
            f"  lr={saved['lr']:.1e}, weight_decay={saved['weight_decay']:.1e}, "
            f"method={saved['method']}, alpha={saved['alpha']:.1f}"
        )
        print(f"  mean TEST AUQC = {saved['mean_AUQC']:.4f} +- {saved['std_AUQC']:.4f}")
        print(f"  saved_at = {saved['saved_at']}")
    else:
        print("No cached best_cfg found in memory and no saved result file exists yet.")
        print("Run Cell 10 once to generate and persist result files.")

No cached best_cfg found in memory and no saved result file exists yet.
Run Cell 10 once to generate and persist result files.
